# Logits Extractor Module

In [ ]:
def default_params(): 
    return {
        'current_model': 'M1',
        'gpu': False,
        'quantization': 'none', #['none',"int4", "int8", "float32", "float16"]
        'dataset': {
            'path': '/workspaces/CodeSmells/datax/code_smells/generation',
            'decoding_strategy': 'curated',
            'content_column': 'code',
            'sampling_size': 500,
        },
        'logging_path': '/workspaces/CodeSmells/datax/code_smells/logs', 
        'callbacks_dir' : '/workspaces/CodeSmells/datax/code_smells/callbacks/generation',
        'cache_dir': '/workspaces/CodeSmells/datax/hugging_face_cache',
        'causal_models': {
            'M1' : 'codellama/CodeLlama-7b-hf', #https://huggingface.co/codellama/CodeLlama-7b-hf, 
            'M2' : 'mistralai/Mistral-7B-v0.3', #https://huggingface.co/mistralai/Mistral-7B-v0.3,
            'M3' : 'microsoft/Phi-3.5-mini-instruct', #https://huggingface.co/microsoft/Phi-3.5-mini-instruct 
            'M4' : 'Qwen/Qwen2.5-Coder-7B', #https://huggingface.co/Qwen/Qwen2.5-Coder-7B
            'M5' : 'facebook/incoder-6B', #https://huggingface.co/facebook/incoder-6B
            'M6' : 'bigcode/starcoder2-7b', #https://huggingface.co/bigcode/starcoder2-7b 
            'M7' : 'deepseek-ai/DeepSeek-R1-Distill-Llama-8B', #https://huggingface.co/deepseek-ai/DeepSeek-R1-Distill-Llama-8B
            'M8' : 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B', #https://huggingface.co/deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B
        },
    }
params = default_params()


#### Imports

In [6]:
import pandas as pd
import os
import time
import numpy as np
import torch
import gc
import seaborn as sns
from scipy import stats
from statistics import NormalDist
import matplotlib.pyplot as plt

In [7]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

In [8]:
def create_folder(path):
    if not os.path.exists(path):
        os.makedirs(path)

In [ ]:
# Define log file path
log_file = f"{params['logging_path']}/{params['current_model']}/{params['dataset']['decoding_strategy']}"
create_folder(log_file)
log_file += '/log.txt'

# Create the log file if it doesn't exist
if not os.path.exists(log_file):
    with open(log_file, 'w'): 
        pass  # Create an empty log file

In [10]:
import logging
logging.basicConfig(filename=log_file, format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)

#### GPU

In [11]:
! nvidia-smi

Tue Feb 18 19:23:31 2025       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 470.103.01   Driver Version: 470.103.01   CUDA Version: 12.3     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  NVIDIA A100-PCI...  Off  | 00000000:61:00.0 Off |                    0 |
| N/A   33C    P0    34W / 250W |      0MiB / 40536MiB |      0%      Default |
|                               |                      |             Disabled |
+-------------------------------+----------------------+----------------------+
                                                                               
+-------

In [12]:
torch.__version__

'2.1.2+cu121'

In [13]:
device = torch.device("cuda:0" if torch.cuda.is_available() and params['gpu'] else "cpu")
device

device(type='cuda', index=0)

In [14]:
torch.cuda.memory_allocated()

0

## Logits Extractor
>
> Extracting Tensor Logits from a given Neural Code Model
>

#### Model Loading

In [15]:
def instantiate_llm(model_name:str, cache_dir:str):
     '''Instantiate AutoModelForCausalLM'''
     tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir = cache_dir)
     logging.info("Loaded AutoTokenizer - " + model_name)
     model = None
     if params['quantization'] == 'int4':
          model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, load_in_4bit=True)
     elif params['quantization'] == 'int8':
          model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, load_in_8bit=True)
     elif params['quantization'] == 'float32':
          model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, torch_dtype=torch.float32)
     elif params['quantization'] == 'float16':
          model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, torch_dtype=torch.float16)
     else: 
          model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir)
     logging.info("Loaded AutoModelForCausalLM - " + model_name)

     return tokenizer, model

In [16]:
tokenizer, model = instantiate_llm(params['causal_models'][params['current_model']], params['cache_dir'])

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [17]:
model.config

LlamaConfig {
  "_name_or_path": "codellama/CodeLlama-7b-hf",
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 1,
  "eos_token_id": 2,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 11008,
  "max_position_embeddings": 16384,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 32,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_scaling": null,
  "rope_theta": 1000000,
  "tie_word_embeddings": false,
  "torch_dtype": "bfloat16",
  "transformers_version": "4.36.2",
  "use_cache": true,
  "vocab_size": 32016
}

In [18]:
model.to(device) #WARNING, Verify the device before assigning to memory

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32016, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaSdpaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (v_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=11008, bias=False)
          (up_proj): Linear(in_features=4096, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm()
        (post_attention_layernorm): LlamaRMSNorm()
      )
    )
    (norm): LlamaRMSNorm()
  )
  (lm_head):

#### Dataset

In [ ]:
df_dataset = pd.read_json(f"{params['dataset']['path']}/{params['dataset']['decoding_strategy']}_{params['dataset']['sampling_size']}.json")

In [21]:
df_dataset

,id,commit_id,repo,path,file_name,fun_name,commit_message,code,url,language,...,n_ast_nodes,n_identifiers,s_msg_id,s_line,s_column,s_end_line,s_end_column,s_code,category,input_lenght
0,256261,a59bca366174d9c692fa19750c24d65f47660ef7,haystack,haystack/modeling/training/base.py,base.py,_get_state_dict,Apply black formatting (#2115)\n\n* Testing bl...,def _get_state_dict(self):\n \n ...,https://github.com/deepset-ai/haystack.git,Python,...,193,20,W0311,2,0,2,22,state_dict = {,Warning,296
1,305801,6f564e4f514b56bce281ec7e82703cfbff87b417,core,homeassistant/components/ring/binary_sensor.py,binary_sensor.py,async_added_to_hass,Improve entity type hints [r] (#77874),async def async_added_to_hass(self) -> None:\n...,https://github.com/home-assistant/core.git,Python,...,63,6,W0311,4,0,4,37,self._dings_update_callback(),Warning,73
2,70649,5fe901e5d86ed02dbbb63039a897582951266afd,wagtail,wagtail/admin/tests/pages/test_edit_page.py,test_edit_page.py,test_new_comment,Fix commenting thread notifications being sent...,def test_new_comment(self):\n post_data...,https://github.com/wagtail/wagtail.git,Python,...,565,37,C0301,33,0,33,125,self.assertEqual(mail.outbox[0].subjec...,Convention,666
3,151753,bdfedb5fcb02b88c600ef25c88bbb5d939b8bd0a,freqtrade,freqtrade/freqai/RL/BaseReinforcementLearningM...,BaseReinforcementLearningModel.py,__init__,Improve typehints / reduce warnings from mypy,"def __init__(self, **kwargs) -> None:\n ...",https://github.com/freqtrade/freqtrade.git,Python,...,411,38,W0311,13,0,13,44,import_str = 'stable_baselines3',Warning,494
4,3868,2282a4ae0221b1fb88e16eca8bc14a166998d2d2,airbyte,airbyte-integrations/connectors/source-hubspot...,streams.py,state,🎉 Source Hubspot: Migrate to CDK (#10177)\n\n*...,"def state(self, value):\n state_value =...",https://github.com/airbytehq/airbyte.git,Python,...,99,14,W0311,7,0,7,61,"self._start_date = max(self._state, se...",Warning,110
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1702546,43471,09f38ad3f6872bae5059a1de226362eb358c4a7a,airflow,tests/providers/microsoft/azure/operators/test...,test_asb.py,test_send_message_queue,Implement Azure Service Bus Queue Operators (#...,"def test_send_message_queue(self, mock_get_con...",https://github.com/apache/airflow.git,Python,...,132,21,C2801,10,12,13,24,mock.call()\n .__enter__()\n ...,Convention,193
1704801,196984,4a6d5d342e1d0111130d4b31708535b862bfacd0,sympy,sympy/printing/repr.py,repr.py,_print_Permutation,Update the deprecation for Permutation.print_c...,"def _print_Permutation(self, expr):\n f...",https://github.com/sympy/sympy.git,Python,...,388,30,C2801,20,16,20,53,Cycle(expr)(expr.size - 1).__repr__(),Convention,441
1705975,105,0b8a53bd313abdf484a9d1e3fbd6aad13c0ec857,PySyft,packages/syft/tests/syft/core/tensor/passthrou...,passthrough_test.py,test__rshift__,adding tests,def test__rshift__() -> None:\n data_a = np...,https://github.com/OpenMined/PySyft.git,Python,...,183,17,C2801,7,15,7,44,tensor_a.__rshift__(tensor_b),Convention,204
1717859,117464,9ce5a21dd6359fd7e8ebf78051ce9e97bd195ec9,mindsdb,tests/unit/executor_test_base.py,executor_test_base.py,set_executor,ML handler supbrocess (#3377)\n\n* log -> logg...,"def set_executor(self, to_mock_model_controlle...",https://github.com/mindsdb/mindsdb.git,Python,...,422,53,C2801,49,27,49,51,config_patch.__enter__(),Convention,610


In [ ]:
#df_dataset = df_dataset[df_dataset['input_lenght']>=700]
#df_dataset = df_dataset[:20]

#### Logit Inference

In [ ]:
def logit_extractor(model, batch, tf_encoded_inputs, from_index=0):
    """
    Output is the class CausalLMOutputWithPast (https://huggingface.co/transformers/v4.10.1/main_classes/output.html?highlight=causallmoutputwithpast)"
    logits (torch.FloatTensor of shape (batch_size, sequence_length, config.vocab_size)) – Prediction scores of the language modeling head (scores for each vocabulary token before SoftMax).
    The expression i.type(torch.LongTensor).to(device) is for casting labels for the loss
    """
    callbacks_dir = f"{params['callbacks_dir']}/{params['current_model']}_q_{params['quantization']}/{params['dataset']['decoding_strategy']}"
    create_folder(callbacks_dir)
    
    for idx, n in enumerate(range(from_index, len(tf_encoded_inputs), batch)):
        torch.cuda.empty_cache()
        output = []
        for encoded_sample in tf_encoded_inputs[n:n+batch]:
            output.append( 
                model(input_ids = encoded_sample, labels = encoded_sample)
            )
        output_logits = [ o['logits'].detach().to('cpu').numpy() for o in output ]  #Logits Extraction
        output_loss = np.array([ o.loss.detach().to('cpu').numpy() for o in output ])  #Language modeling loss (for next-token prediction).

        #Saving Callbacks
        current_batch = idx + (from_index//batch)
        for jdx, o_logits in enumerate(output_logits):
            np.save(f"{callbacks_dir}/logits_tensor[{jdx+n}]_batch[{current_batch}].npy", o_logits)
        np.save(f"{callbacks_dir}/_loss_batch[{current_batch}].npy", output_loss)
        
        print(f"Batch [{current_batch}] Completed")

        #Memory Released
        for out in output:
            del out.logits
            torch.cuda.empty_cache()
            del out.loss
            torch.cuda.empty_cache()
        for out in output_logits:
            del out
            torch.cuda.empty_cache()
        for out in output_loss:
            del out
            torch.cuda.empty_cache()

In [26]:
#Casting Integers to Tensor Integers. Make sure the tesor is created in a device
#We ignored the parameter attention_mask since we are not using masking here [https://huggingface.co/transformers/v4.10.1/glossary.html#attention-mask]
tf_encoded_inputs = [tokenizer(sample, return_tensors='pt')['input_ids'].to(device) for sample in df_dataset[params['dataset']['content_column']].values]

In [27]:
## ACTUAL EXPERIMENT
## TIME AND MEMORY CONSUMING
logit_extractor(
    model = model,
    batch = 1, 
    tf_encoded_inputs = tf_encoded_inputs, 
    from_index=0
)

Batch [0] Completed
Batch [1] Completed
Batch [2] Completed
Batch [3] Completed
Batch [4] Completed
Batch [5] Completed
Batch [6] Completed
Batch [7] Completed
Batch [8] Completed
Batch [9] Completed
Batch [10] Completed
Batch [11] Completed
Batch [12] Completed
Batch [13] Completed
Batch [14] Completed
Batch [15] Completed
Batch [16] Completed
Batch [17] Completed


In [ ]:
print("================================= PROCESS COMPLETE =================================")

In [ ]:
del model
torch.cuda.empty_cache()
gc.collect()

: 